<div dir="rtl" align="right">

# جنگل تصادفی برای رگرسیون

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
یک درخت تنها ناپایدار است، ولی **میانگین صدها درخت متفاوت** پایدار و دقیق است. مثل اینکه به جای پرسیدن از یک نفر، از جمع زیادی بپرسی و میانگین بگیری (**خرد جمعی**).

## دو منبع تصادف
- **Bagging:** هر درخت روی یک **نمونه بوت‌استرپ** آموزش می‌بیند: n نمونه با جایگذاری از داده برمی‌داریم. حدود ۶۳٪ نمونه‌ها در هر بوت‌استرپ هستند و بقیه (**خارج از کیسه**، OOB) برای ارزیابی رایگان استفاده می‌شوند.
- **انتخاب تصادفی ویژگی:** در هر گره فقط زیرمجموعه تصادفی از ویژگی‌ها بررسی می‌شود (`max_features`). این کار درخت‌ها را از هم **مستقل‌تر** می‌کند.

$$\hat y(x)=\frac1B\sum_{b=1}^{B}T_b(x)$$

## چرا میانگین کمک می‌کند؟
اگر $B$ پیش‌بینی با واریانس $\sigma^2$ و همبستگی $\rho$ داشته باشیم، واریانس میانگین:

$$\text{Var}=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$$

با زیاد کردن $B$ جمله دوم صفر می‌شود؛ با کم کردن همبستگی $\rho$ (ویژگی تصادفی) جمله اول هم کم می‌شود.

> [۱ درخت: پله‌ای و پرنویز. ۱۰ درخت: بهتر. ۲۰۰ درخت: صاف و پایدار.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/forest_reg_n.png)

*۱ درخت: پله‌ای و پرنویز. ۱۰ درخت: بهتر. ۲۰۰ درخت: صاف و پایدار.*

[[lab:reg_forest]]

در آزمایشگاه خطوط کم‌رنگ درخت‌های تکی‌اند؛ خط پررنگ میانگین آن‌هاست. تعداد درخت را از ۱ زیاد کن.

## هایپرپارامترها
| هایپرپارامتر | پیش‌فرض | توضیح |
|---|---|---|
| `n_estimators` | 100 | تعداد درخت؛ بیشتر بهتر ولی کندتر (بیش‌برازش ایجاد نمی‌کند) |
| `max_features` | 1.0 در رگرسیون | سهم ویژگی‌ها در هر گره؛ 0.3 تا 0.5 را امتحان کن |
| `max_depth` / `min_samples_leaf` | بدون محدودیت / 1 | پیچیدگی هر درخت |
| `oob_score` | False | ارزیابی رایگان با نمونه‌های خارج از کیسه |



</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

X, y = load_diabetes(return_X_y=True)
for n in [1, 10, 50, 200]:
    rf = RandomForestRegressor(n_estimators=n, min_samples_leaf=5, max_features=0.5, oob_score=n > 20, random_state=0)
    cv = cross_val_score(rf, X, y, cv=5, scoring="r2").mean()
    rf.fit(X, y)
    oob = f"{rf.oob_score_:.3f}" if n > 20 else "-"
    print(f"{n:4d} trees: CV R2 = {cv:.3f}   OOB R2 = {oob}")

# Share of unique samples in one bootstrap sample
rng = np.random.default_rng(0)
print("unique share:", len(np.unique(rng.integers(0, 1000, 1000))) / 1000)

<div dir="rtl" align="right">



## اهمیت ویژگی‌ها
جنگل می‌گوید هر ویژگی چقدر خطا را کم کرده است. ولی این اهمیت برای ویژگی‌هایی با مقادیر یکتای زیاد سوگیری دارد؛ **اهمیت جایگشتی** (`permutation_importance`) قابل اعتمادتر است:



</div>

In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

d = load_diabetes(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(d.data, d.target, random_state=0)
rf = RandomForestRegressor(300, min_samples_leaf=5, random_state=0).fit(X_tr, y_tr)
imp = pd.DataFrame({"impurity": rf.feature_importances_,
                    "permutation": permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0).importances_mean},
                   index=d.data.columns)
print(imp.sort_values("permutation", ascending=False).round(3))

<div dir="rtl" align="right">



| مزایا | معایب |
|---|---|
| دقیق با تنظیم کم | تفسیر سخت‌تر از یک درخت |
| مقاوم در برابر بیش‌برازش | حجم و زمان پیش‌بینی بیشتر |
| بدون نیاز به مقیاس‌بندی | برون‌یابی نمی‌کند |
| ارزیابی OOB رایگان | |

</div>

<div dir="rtl" align="right">

### تمرین

max_features را 0.2، 0.5 و 1.0 بگذار و R² اعتبارسنجی را مقایسه کن.

</div>

In [ ]:
# Your answer here